# Phase 12B — SAR Training (NoSQL)

Trains the **Schema-Aware Retriever** on the NoSQL (MongoDB MQL) corpus.
Identical process to Phase 12A (SQL) — only the corpus and output path change.

| | Detail |
|---|---|
| **Model trained** | `SchemaAwareModel` (~16M params) — same architecture as SQL SAR |
| **Model NOT trained** | BGE (BAAI/bge-large-en-v1.5) — frozen encoder |
| **Input data** | `Data/rag_corpus/spider_nosql_rag.json` — 5697 entries |
| **Structural types** | 52 unique types (vs 57 for SQL) |
| **Output** | `sar_model.pt` (~50 MB) saved to Google Drive |
| **Est. training time** | ~1–2 minutes on T4 |

## NoSQL corpus vs SQL corpus

| | SQL | NoSQL |
|---|---|---|
| Corpus file | `spider_sql_rag.json` | `spider_nosql_rag.json` |
| Entries | 7000 | 5697 |
| Structural types | 57 | 52 |
| Query field | `sql` | `mql_pipeline` |
| Structural type source | Parsed from SQL | Converted from SQL via `source_sql` |

The NoSQL corpus has the same `structural_type` format as SQL (num_joins, has_group_by, etc.) because the MongoDB pipelines were converted from Spider SQL queries. SAR uses only `structural_type` and `question` — the query format (SQL vs MQL) does not affect training.

## Cell 1 — Verify GPU

In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout)

gpu = result.stdout
if 'T4' in gpu:
    print('T4 detected — NoSQL SAR trains in ~1-2 min on T4 ✅')
elif 'A100' in gpu:
    print('A100 detected ✅')
else:
    print('⚠️  Unknown GPU — check Runtime → Change runtime type')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sar_nosql'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR} ✅')

## Cell 3 — Clone or update repo

In [ ]:
%%bash
set -e

REPO_DIR="/content/Codegen"
BRANCH="phase/12b-sar-nosql"

if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo exists — pulling latest"
    cd "$REPO_DIR"
    git fetch origin
    git checkout $BRANCH
    git pull origin $BRANCH
else
    echo "Cloning repo"
    git clone https://github.com/kethansplunk/Codegen.git "$REPO_DIR"
    cd "$REPO_DIR"
    git checkout $BRANCH
fi

echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 4 — Install dependencies

In [ ]:
!pip install -q FlagEmbedding scikit-learn
print('Dependencies installed ✅')

## Cell 5 — Inspect NoSQL corpus

The NoSQL corpus was built in Phase 8B. Each entry has:
- `question`: natural language question
- `mql_pipeline`: the MongoDB aggregation pipeline
- `mql_collection`: the primary collection name
- `structural_type`: same 6-dim vector as SQL (derived from `source_sql`)
- `source_sql`: the original Spider SQL it was converted from

SAR only uses `question` and `structural_type` — the MQL format is irrelevant for training.

In [ ]:
import json
from collections import Counter

CORPUS_PATH = '/content/Codegen/Data/rag_corpus/spider_nosql_rag.json'

with open(CORPUS_PATH) as f:
    corpus = json.load(f)

print(f'Total entries     : {len(corpus)}')
print(f'Keys per entry    : {list(corpus[0].keys())}')
print(f'Sample question   : {corpus[0]["question"]}')
print(f'Sample MQL        : {corpus[0].get("mql_pipeline", "N/A")[:80]}...')
print(f'Structural type   : {corpus[0]["structural_type"]}')

def st_key(st): return tuple(sorted(st.items()))
counts = Counter(st_key(d['structural_type']) for d in corpus)
print(f'\nUnique structural types : {len(counts)}')
print('Top 5 most common types:')
for k, v in counts.most_common(5):
    d = dict(k)
    flags = [name for name, val in d.items() if val is True]
    joins = d.get('num_joins', 0)
    desc  = f"{v:4d} entries | joins={joins}" + (f" | {', '.join(flags)}" if flags else " | simple lookup")
    print(f'  {desc}')

## Cell 6 — Run SAR NoSQL training

**Phase A — BGE encoding (~1 min)**
- Encodes all 5697 NoSQL questions into 1024-dim vectors
- Cached to `Data/sar_nosql_emb_cache.pkl` — skipped on re-run

**Phase B — Triplet training (~1 min, 10 epochs)**
- Same SchemaAwareModel architecture as Phase 12A
- Trained from scratch on NoSQL structural types
- Loss should drop from ~0.14 → ~0.02 (same pattern as SQL)

> **Note:** This is a separate model from the SQL SAR. At inference time, SQL questions use
> `sar_sql/sar_model.pt` and NoSQL questions use `sar_nosql/sar_model.pt`.

In [ ]:
%%bash
cd /content/Codegen

echo "=== Phase A: BGE embedding (cached after first run) ==="
echo "=== Phase B: SchemaAwareModel triplet training        ==="
echo ""

python -m src.sar.train \
    --corpus Data/rag_corpus/spider_nosql_rag.json \
    --out    /content/drive/MyDrive/codegen/checkpoints/sar_nosql \
    --epochs 10 \
    --batch  32 \
    --lr     1e-4 \
    --margin 0.3

## Cell 7 — Verify output

In [ ]:
import os

OUT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sar_nosql'

print('Files in output directory:')
for fname in sorted(os.listdir(OUT_DIR)):
    fpath   = os.path.join(OUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<40} {size_mb:>8.1f} MB')

model_path = os.path.join(OUT_DIR, 'sar_model.pt')
if os.path.exists(model_path):
    size = os.path.getsize(model_path) / 1e6
    print(f'\nsar_model.pt size: {size:.1f} MB  (expected ~50 MB) ✅')
    print('SAR NoSQL training complete — both SQL and NoSQL SAR models ready')
else:
    print('\n⚠️  sar_model.pt not found — training may still be running or failed')

## Cell 8 — Smoke test (optional)

Retrieves top-3 structurally similar NoSQL examples for a sample question.
The results should be other questions that need the same MongoDB pipeline pattern.

In [ ]:
import sys
sys.path.insert(0, '/content/Codegen')

from src.sar.infer import SARRetriever

MODEL_PATH  = '/content/drive/MyDrive/codegen/checkpoints/sar_nosql/sar_model.pt'
CORPUS_PATH = '/content/Codegen/Data/rag_corpus/spider_nosql_rag.json'

retriever = SARRetriever(
    model_path=MODEL_PATH,
    corpus_path=CORPUS_PATH,
)

# GROUP BY question — should retrieve other GROUP BY / $group pipeline examples
test_question = "How many products are there in each category?"
results = retriever.retrieve(test_question, top_k=3)

print(f'Query: "{test_question}"')
print(f'\nTop-3 structurally similar NoSQL examples:')
for i, r in enumerate(results, 1):
    print(f'\n  [{i}] Question   : {r["question"]}')
    print(f'       Collection : {r.get("mql_collection", "N/A")}')
    print(f'       Pipeline   : {str(r.get("mql_pipeline", ""))[:100]}...')
    print(f'       Struct type: {r["structural_type"]}')